In [1]:
# ============================================================
# HyDE (Hypothetical Document Embeddings) - Complete Example
# ============================================================

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# ------------------------------------------------------------
# Step 1 : Sample Knowledge Base
# ------------------------------------------------------------

docs = [
    Document(page_content="LangChain is a framework for building LLM applications."),
    Document(page_content="FAISS is a vector database used for semantic search."),
    Document(page_content="Prompt Engineering improves LLM responses."),
    Document(page_content="RAG combines retrieval with LLMs."),
    Document(page_content="Conversation Memory helps chatbots remember previous interactions."),
]



c:\Users\SHAILENDRA\anaconda3\envs\env_langchain\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\SHAILENDRA\AppData\Local\Temp\ipykernel_8436\4251597236.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
# ------------------------------------------------------------
# Step 2 : Create Vector Store
# ------------------------------------------------------------

embedding = OpenAIEmbeddings()

vectorstore = FAISS.from_documents(
    docs,
    embedding
)

retriever = vectorstore.as_retriever()

# ------------------------------------------------------------
# Step 3 : Initialize LLM
# ------------------------------------------------------------

llm = init_chat_model(
    "openai:gpt-4.1-mini",
    temperature=0
)



In [3]:
# ------------------------------------------------------------
# Step 4 : HyDE Prompt
# ------------------------------------------------------------

hyde_prompt = ChatPromptTemplate.from_messages(
[
    (
        "system",
        """
You are an expert technical writer.

Generate a detailed hypothetical document that would perfectly answer
the user's question.

This document is ONLY used for semantic retrieval.

Do not say it is hypothetical.
"""
    ),
    (
        "human",
        "{question}"
    )
]
)

# ------------------------------------------------------------
# Step 5 : HyDE Generation Chain
# ------------------------------------------------------------

hyde_chain = (
    hyde_prompt
    | llm
    | StrOutputParser()
)

# ------------------------------------------------------------
# Step 6 : Helper Function
# ------------------------------------------------------------



In [4]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# ------------------------------------------------------------
# Step 7 : Retrieve using HyDE Document
# ------------------------------------------------------------



In [5]:
def retrieve_context(x):

    # Original Question
    question = x["question"]

    print("="*80)
    print("USER QUESTION")
    print("="*80)
    print(question)

    # Generate hypothetical document
    hypothetical_doc = hyde_chain.invoke(
        {
            "question": question
        }
    )

    print("\n")
    print("="*80)
    print("HYPOTHETICAL DOCUMENT")
    print("="*80)
    print(hypothetical_doc)

    # Semantic Search
    retrieved_docs = retriever.invoke(
        hypothetical_doc
    )

    print("\n")
    print("="*80)
    print("RETRIEVED DOCUMENTS")
    print("="*80)

    for i, doc in enumerate(retrieved_docs, 1):
        print(f"\nDocument {i}")
        print(doc.page_content)

    return format_docs(retrieved_docs)

# ------------------------------------------------------------
# Step 8 : Final RAG Prompt
# ------------------------------------------------------------

answer_prompt = ChatPromptTemplate.from_messages(
[
    (
        "system",
        """
Answer ONLY using the provided context.

If the answer cannot be found,
say you don't know.
"""
    ),
    (
        "human",
        """
Context:
{context}

Question:
{question}
"""
    )
]
)



In [6]:
# ------------------------------------------------------------
# Step 9 : Complete HyDE RAG Pipeline
# ------------------------------------------------------------

hyde_rag_chain = (

    RunnableMap(
    {
        "context": retrieve_context,
        "question": lambda x: x["question"]
    }
    )

    | answer_prompt
    | llm
    | StrOutputParser()

)



In [7]:
# ------------------------------------------------------------
# Step 10 : Run
# ------------------------------------------------------------

response = hyde_rag_chain.invoke(
{
    "question":"How can I build an AI chatbot?"
}
)

print("\n")
print("="*80)
print("FINAL ANSWER")
print("="*80)

print(response)

USER QUESTION
How can I build an AI chatbot?


HYPOTHETICAL DOCUMENT
# How to Build an AI Chatbot: A Comprehensive Guide

Building an AI chatbot involves several key steps, from defining the chatbot’s purpose to deploying it for users. This guide provides a detailed roadmap to help you create an effective AI chatbot.

---

## 1. Define the Purpose and Scope

Before development, clearly outline what your chatbot will do:

- **Use Case:** Customer support, personal assistant, information retrieval, entertainment, etc.
- **Target Audience:** Who will interact with the chatbot?
- **Platform:** Website, mobile app, messaging platforms (Facebook Messenger, WhatsApp, Slack), or voice assistants.

---

## 2. Choose the Type of Chatbot

- **Rule-Based Chatbots:** Follow predefined rules and scripts. Simple but limited.
- **AI-Powered Chatbots:** Use Natural Language Processing (NLP) and Machine Learning (ML) to understand and respond dynamically.

For a more advanced and flexible chatbot, AI-po